# 04 · Modelo Random Forest

Objetivo: entrenar y ajustar un clasificador Random Forest.

- Búsqueda de hiperparámetros (n_estimators, max_depth, etc.)
- Validación cruzada
- Importancia de características
- Persistencia del modelo entrenado

In [1]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report
)

carpeta_datos = Path("../data/procesado")
carpeta_modelos = Path("../models")
carpeta_modelos.mkdir(parents=True, exist_ok=True)

X_train = np.load(carpeta_datos / "X_train.npy")
X_test = np.load(carpeta_datos / "X_test.npy")
y_train = np.load(carpeta_datos / "y_train.npy")
y_test = np.load(carpeta_datos / "y_test.npy")

columnas_sabores = [
    "dulce", "salado", "amargo", "acido", "umami"
]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (3465, 1814)
X_test: (1486, 1814)
y_train: (3465, 5)
y_test: (1486, 5)


In [2]:

modelo_rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1
)

print("Entrenando Random Forest...")
modelo_rf.fit(X_train, y_train)

joblib.dump(
    modelo_rf,
    carpeta_modelos / "random_forest_multietiqueta.joblib"
)

print("Random Forest entrenado y guardado.")


Entrenando Random Forest...
Random Forest entrenado y guardado.


In [3]:

y_pred_rf = modelo_rf.predict(X_test)

metricas_rf = {
    "F1 macro": f1_score(
        y_test, y_pred_rf, average="macro", zero_division=0
    ),
    "F1 micro": f1_score(
        y_test, y_pred_rf, average="micro", zero_division=0
    ),
    "Precision macro": precision_score(
        y_test, y_pred_rf, average="macro", zero_division=0
    ),
    "Recall macro": recall_score(
        y_test, y_pred_rf, average="macro", zero_division=0
    ),
    "Hamming loss": hamming_loss(y_test, y_pred_rf)
}

print("MÉTRICAS DE RANDOM FOREST")
for nombre, valor in metricas_rf.items():
    print(f"{nombre}: {valor:.4f}")

print("\nREPORTE POR SABOR")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=columnas_sabores,
    zero_division=0
))


MÉTRICAS DE RANDOM FOREST
F1 macro: 0.3359
F1 micro: 0.6679
Precision macro: 0.6068
Recall macro: 0.4045
Hamming loss: 0.2766

REPORTE POR SABOR
              precision    recall  f1-score   support

       dulce       0.65      0.02      0.04       571
      salado       0.76      1.00      0.86      1127
      amargo       0.00      0.00      0.00        67
       acido       1.00      0.00      0.00       509
       umami       0.63      1.00      0.77       927

   micro avg       0.69      0.65      0.67      3201
   macro avg       0.61      0.40      0.34      3201
weighted avg       0.72      0.65      0.53      3201
 samples avg       0.69      0.62      0.64      3201



In [4]:

np.save(
    carpeta_datos / "y_pred_rf.npy",
    y_pred_rf
)

pd.DataFrame([
    {"modelo": "Random Forest", "metrica": nombre, "valor": valor}
    for nombre, valor in metricas_rf.items()
]).to_csv(
    carpeta_datos / "metricas_rf.csv",
    index=False
)

print("Predicciones y métricas guardadas correctamente.")


Predicciones y métricas guardadas correctamente.
